# ATUS TV-Watching Dataset Preparation

Converts an ATUS (American Time Use Survey) activity file into an **hourly binary TV-watching** dataset matching the project format.

**TV activity codes captured** (`trcodep` starting with `1203`):
- `120301` — Watching TV/movies (pre-recorded)
- `120303` — Watching TV (live)
- `120304` — Watching TV/videos on a computer
- `120312` — Streaming video

**Two output modes:**
- `per_respondent` — one row per (respondent, hour); used for multi-household model training
- `population` — averaged across all respondents; used for baseline comparisons

**Usage:**
```bash
python prepare_tv.py --input atusact.csv --output atus_tv_hourly.csv --mode per_respondent
```

## Imports & TV Code Definition

In [9]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

## ATUS Time Parser

In [10]:
# ---------------------------------------------------------------------------
# ATUS activity codes that represent TV watching
# All codes starting with 1203 are "TV and movies"
# ---------------------------------------------------------------------------

TV_CODE_PREFIX = 1203


def parse_atus_time(time_str):
    """Parse HH:MM:SS string from ATUS into a timedelta from midnight."""
    h, m, s = map(int, time_str.strip().split(":"))
    return timedelta(hours=h, minutes=m, seconds=s)

## Data Loader

Loads the ATUS activity CSV and flags TV rows based on the `trcodep` code prefix.

In [11]:
def load_atus(path):
    df = pd.read_csv(path, dtype={"trcodep": int, "tucaseid": str})

    # Keep only the columns we need
    cols = ["tucaseid", "tustarttim", "tustoptime", "tuactdur24", "trcodep"]
    df = df[cols].copy()

    # Flag TV rows
    df["is_tv"] = (df["trcodep"] // 100 == TV_CODE_PREFIX)

    return df

## Activity-to-Hours Expansion

Expands each activity row into per-clock-hour entries, correctly handling ATUS diary days that run 04:00–03:59 (crossing midnight).

In [12]:
def expand_activity_to_hours(row, diary_date):
    
    base = diary_date  # midnight of the diary date

    start_td = parse_atus_time(row["tustarttim"])
    stop_td  = parse_atus_time(row["tustoptime"])

    # ATUS wraps midnight: if stop <= start the activity crosses midnight
    if stop_td <= start_td:
        stop_td += timedelta(days=1)

    start_dt = base + start_td
    stop_dt  = base + stop_td

    hours = []
    current = start_dt.replace(minute=0, second=0, microsecond=0)
    while current < stop_dt:
        next_hour = current + timedelta(hours=1)
        overlap_start = max(current, start_dt)
        overlap_end   = min(next_hour, stop_dt)
        minutes = (overlap_end - overlap_start).total_seconds() / 60
        if minutes > 0:
            hours.append((current, minutes))
        current = next_hour

    return hours

## Hourly Aggregation

Aggregates expanded activity chunks to one row per (respondent, hour) with total `tv_minutes` and a binary `elec_television_on` flag.

In [13]:
def build_hourly(df):
    records = []
    SYNTHETIC_BASE = datetime(2024, 1, 1)   # placeholder; replace if joining respondent file

    tv_rows = df[df["is_tv"]].copy()

    for _, row in tv_rows.iterrows():
        hour_chunks = expand_activity_to_hours(row, SYNTHETIC_BASE)
        for hour_dt, minutes in hour_chunks:
            records.append({
                "tucaseid":   row["tucaseid"],
                "timestamp":  hour_dt,
                "tv_minutes": minutes,
            })

    if not records:
        print("No TV activity rows found. Check that trcodep values start with 1203.")
        return pd.DataFrame()

    long = pd.DataFrame(records)

    # Aggregate: per respondent per hour — total TV minutes and binary on/off
    agg = (
        long.groupby(["tucaseid", "timestamp"])["tv_minutes"]
        .sum()
        .reset_index()
    )
    agg["elec_television_on"] = (agg["tv_minutes"] > 0).astype(int)

    # Add time features to match project format
    agg["hour"]        = agg["timestamp"].dt.hour
    agg["day_of_week"] = agg["timestamp"].dt.dayofweek   # 0=Mon, 6=Sun
    agg["is_weekend"]  = (agg["day_of_week"] >= 5).astype(int)
    agg["month"]       = agg["timestamp"].dt.month
    agg["is_evening"]  = agg["hour"].between(18, 23).astype(int)

    return agg




## Population Aggregation

Averages across all respondents to produce a single population-level hourly time series.

In [14]:
def aggregate_population(hourly_df):
    pop = (
        hourly_df.groupby("timestamp")
        .agg(
            on=("elec_television_on", "mean"),          # fraction of respondents watching
            tv_minutes=("tv_minutes", "mean"),
            hour=("hour", "first"),
            day_of_week=("day_of_week", "first"),
            is_weekend=("is_weekend", "first"),
            month=("month", "first"),
            is_evening=("is_evening", "first"),
        )
        .reset_index()
        .sort_values("timestamp")
    )
    # Binarise the population average at 0.5 threshold to match on/off format
    pop["on_binary"] = (pop["on"] >= 0.5).astype(int)
    return pop

## Run Pipeline

Set `INPUT_PATH`, `OUTPUT_PATH`, and `MODE` below, then run the cell.

In [15]:
# ── Configuration — set your paths and mode here ──────────────────
INPUT_PATH  = "atusact.csv"         # path to ATUS activity CSV
OUTPUT_PATH = "atus_tv_hourly.csv"  # output CSV path
MODE        = "per_respondent"       # "per_respondent" or "population"
# ───────────────────────────────────────────────────────────────────

print(f"Loading {INPUT_PATH} ...")
df = load_atus(INPUT_PATH)
print(f"  Total activity rows : {len(df):,}")
print(f"  TV activity rows    : {df['is_tv'].sum():,}")

print("Expanding activities to hourly slots ...")
hourly = build_hourly(df)

if not hourly.empty:
    print("Filling missing hours with on=0 ...")
    all_hours   = pd.date_range("2024-01-01 04:00", periods=24, freq="h")
    respondents = hourly["tucaseid"].unique()
    full_index  = pd.MultiIndex.from_product(
        [respondents, all_hours], names=["tucaseid", "timestamp"]
    )
    hourly = (
        hourly.set_index(["tucaseid", "timestamp"])
        .reindex(full_index, fill_value=0)
        .reset_index()
    )
    hourly["hour"]        = hourly["timestamp"].dt.hour
    hourly["day_of_week"] = hourly["timestamp"].dt.dayofweek
    hourly["is_weekend"]  = (hourly["day_of_week"] >= 5).astype(int)
    hourly["month"]       = hourly["timestamp"].dt.month
    hourly["is_evening"]  = hourly["hour"].between(18, 23).astype(int)

    if MODE == "population":
        result = aggregate_population(hourly)
        print(f"  Population-level rows: {len(result):,}")
    else:
        result = hourly
        print(f"  Per-respondent rows: {len(result):,}")

    result.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved -> {OUTPUT_PATH}")
    print(result.head(10).to_string(index=False))
    print(f"\nTV-on fraction overall : {result['elec_television_on'].mean():.2%}")
    print(f"Peak hour (by on rate) : {result.groupby('hour')['elec_television_on'].mean().idxmax():02d}:00")

Loading atusact.csv ...
  Total activity rows : 3,347,093
  TV activity rows    : 424,241
Expanding activities to hourly slots ...
Filling missing hours with on=0 ...
  Per-respondent rows: 3,727,872

Saved -> atus_tv_hourly.csv
      tucaseid           timestamp  tv_minutes  elec_television_on  hour  day_of_week  is_weekend  month  is_evening
20030100013280 2024-01-01 04:00:00         0.0                   0     4            0           0      1           0
20030100013280 2024-01-01 05:00:00         0.0                   0     5            0           0      1           0
20030100013280 2024-01-01 06:00:00         0.0                   0     6            0           0      1           0
20030100013280 2024-01-01 07:00:00         0.0                   0     7            0           0      1           0
20030100013280 2024-01-01 08:00:00         0.0                   0     8            0           0      1           0
20030100013280 2024-01-01 09:00:00         0.0                   0   